# Live Sensor + TinyML\nTrain a small light classifier and export it for ESP32 TensorFlow Lite Micro.

In [ ]:
import numpy as np
import tensorflow as tf

x = np.array([[0.05],[0.15],[0.25],[0.40],[0.50],[0.60],[0.70],[0.85],[0.95]], dtype=np.float32)
y = np.array([0,0,0,1,1,1,2,2,2], dtype=np.int32)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x, y, epochs=250, verbose=0)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
open('light_level_classifier.tflite', 'wb').write(tflite_model)

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
labels = ['Dark', 'Normal', 'Bright']
for value in [0.10, 0.50, 0.90]:
    interpreter.set_tensor(inp['index'], np.array([[value]], dtype=np.float32))
    interpreter.invoke()
    pred = int(np.argmax(interpreter.get_tensor(out['index'])[0]))
    print(f'{value:.2f} -> {labels[pred]}')


## Export for ESP32\nAfter this notebook creates `light_level_classifier.tflite`, run `python convert_model.py` to generate `model.h`. Copy that generated header into the Arduino project.